# P4-A2 — Human-in-the-Loop (approve before acting)

Back in P2-A6 you flagged the key agent safeguard: *pause for confirmation before a destructive action.* Now you build it. LangGraph's `interrupt_before` stops the graph **right before the tools node** — the agent has *decided* to call a tool but hasn't run it yet. You inspect the pending action, then **approve** (resume) or **reject** (don't).

This is built directly on P4-A1: the interrupt only works because the checkpointer can freeze the state mid-run and resume it later.

In [ ]:
# --- Setup (given) ---
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini')

ORDERS = {'A123': {'status': 'shipped'}, 'B456': {'status': 'processing'}}

@tool
def get_order_status(order_id: str) -> dict:
    """Get the status of an order by ID (safe, read-only)."""
    return ORDERS.get(order_id, {'status': 'not_found'})

@tool
def cancel_order(order_id: str) -> dict:
    """Cancel an order by ID (DESTRUCTIVE — changes state)."""
    if order_id in ORDERS:
        ORDERS[order_id]['status'] = 'cancelled'
        return {'order_id': order_id, 'result': 'cancelled'}
    return {'order_id': order_id, 'result': 'not_found'}

TOOLS = [get_order_status, cancel_order]

def build_agent():
    """Agent that PAUSES before running any tool (interrupt_before=['tools'])."""
    bound = llm.bind_tools(TOOLS)
    def agent_node(state):
        return {'messages': [bound.invoke(state['messages'])]}
    g = StateGraph(MessagesState)
    g.add_node('agent', agent_node)
    g.add_node('tools', ToolNode(TOOLS))
    g.add_edge(START, 'agent')
    g.add_conditional_edges('agent', tools_condition)
    g.add_edge('tools', 'agent')
    return g.compile(checkpointer=InMemorySaver(), interrupt_before=['tools'])

def pending_tool_calls(agent, config):
    """Helper: if the agent is paused before tools, return the tool calls it WANTS to run."""
    state = agent.get_state(config)
    if not state.next:                      # state.next is empty if not interrupted
        return None
    return state.values['messages'][-1].tool_calls

print('ready')

## Task 1 — Run, and watch it PAUSE before acting
Invoke the agent with *"Please cancel order B456."* and a `thread_id` config. Because of `interrupt_before=['tools']`, it stops before executing. Use `pending_tool_calls(agent, config)` to print the tool call it's waiting to run, and confirm `ORDERS['B456']` is **still 'processing'** (nothing happened yet).
<br>Hint: `agent = build_agent()`; `config = {'configurable': {'thread_id': 't1'}}`; `agent.invoke({'messages':[{'role':'user','content':'...'}]}, config=config)`.

In [ ]:
# TODO: invoke the cancel request; show the pending tool call + that ORDERS is unchanged


## Task 2 — APPROVE: resume and let it run
The human approves. Resume the graph by invoking with `None` as the input and the **same config** — LangGraph continues from where it paused, runs the tool, and finishes. Print the final answer and confirm `ORDERS['B456']` is now `'cancelled'`.
<br>Hint: `agent.invoke(None, config=config)` resumes a paused graph.

In [ ]:
# TODO: approve -> resume with invoke(None, config) -> verify the order is cancelled


## Task 3 — REJECT: decline and don't run it
Now the safety case. On a **fresh thread_id**, ask to cancel `A123`. It pauses again. This time the human says **no** — so you simply *don't* resume. Show that `ORDERS['A123']` is **still 'shipped'** (the destructive action never executed).
<br>Goal: the agent proposed an action; the human had the final say and blocked it. That's the whole point of HITL.
<br>(Bonus: print a message back to the 'user' explaining the action was declined.)

In [ ]:
# TODO: fresh thread; pause on cancel A123; DON'T resume; show A123 unchanged


## Task 4 — Explain (in your own words)
1. Why is human-in-the-loop important specifically for *agents* (vs. a plain chatbot)? What class of tool should always gate behind it?
2. How does the interrupt/resume mechanism rely on what you built in P4-A1? (What makes 'pause now, resume later' even possible?)

> _your answer here_